## Text input

https://platform.openai.com/docs/models

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [6]:
from langchain.agents import create_agent

agent = create_agent(
    model='gpt-5-nano',
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
)

In [7]:
from langchain.messages import HumanMessage

question = HumanMessage(content=[
    {"type": "text", "text": "What is the capital of The Moon?"}
])

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

The capital is Lunaris Prime, often spoken of as Lunopolis by the residents of The Moon.

Quick worldbuilding snapshot:
- Location: A ring of arcologies perched along the rim of Nyx Crater near the equator, with direct sightlines to Earth.
- Government: The political heart of the Lunar Commonwealth, home to the Prime Regent and the Legislative Concourse that coordinates settlements across the Moon.
- Architecture: A honeycomb of basalt towers, glass-domed parks, and solar-canopy promenades. The city uses self-healing materials and a lattice of skybridges to connect districts.
- Economy and daily life: Powered by solar farms and compact micro-reactors; ice mined from polar deposits supplies water; hydroponic farms feed the city.
- Landmarks: The Beacon Tower for Earthrise signals, the Earthview Promenade along the crater rim, and the Crescent Gate that serves as the ceremonial entrance to Lunaris Prime.


## Image input

In [2]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [3]:
print(uploader.value)

({'name': 'lunar_city.png', 'type': 'image/png', 'size': 73890, 'content': <memory at 0x10f582980>, 'last_modified': datetime.datetime(2026, 3, 10, 17, 32, 4, 364000, tzinfo=datetime.timezone.utc)},)


In [4]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [8]:
multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this capital"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

The capital pictured is Lunara, the gleaming heartbeat of the Lunar Confederacy. It sits in a shielded crater on the Moon’s edge, a city built under a transparent dome that lets in starlight while keeping the air, temperature, and gravity just right for life.

Key features
- Dome and climate: A vast, transparent habitat dome protects Lunara from vacuum, micrometeoroids, and solar radiation. Inside, a controlled microclimate, air recycling, and hydroponic farms create a comfortable, Earthlike environment despite the Moon’s harsh surface.
- Skyline and districts: A skyline of glass and alloy towers rises from the crater floor. Each district is tiered: citizen housing around the base, midtown offices and markets, and a celestial-view observatory ring near the top. The city feels like a jewel encased in a bubble.
- Energy and tech: Lunara runs on compact fusion reactors and sun-trapping photovoltaics on the dome’s exterior. Regolith-tested recycling, closed-loop water systems, and smart-gl

## Audio input

In [9]:
import sounddevice as sd
from scipy.io.wavfile import write
import base64
import io
import time
from tqdm import tqdm

# Recording settings
duration = 5  # seconds
sample_rate = 44100

print("Recording...")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)
# Progress bar for the duration
for _ in tqdm(range(duration * 10)):   # update 10× per second
    time.sleep(0.1)
sd.wait()
print("Done.")

# Write WAV to an in-memory buffer
buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()

aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")

Recording...


100%|██████████| 50/50 [00:05<00:00,  9.63it/s]


Done.


In [10]:
agent = create_agent(
    model='gpt-4o-audio-preview',
)

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this audio file"},
    {"type": "audio", "base64": aud_b64, "mime_type": "audio/wav"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

The audio file appears to contain someone reciting a short poem or piece of writing about cats. The speaker sounds calm and expressive, describing the nature of cats or perhaps sharing a poetic perspective on them. There’s a clear focus on cats as the subject, but I can’t identify any personal details about the speaker. If you would like a transcription or more detail about the content, let me know!
